## Vehicle Type + tracking

In [ ]:
# from ultralytics import YOLO
# import cv2
# import numpy as np
# from collections import defaultdict
# from datetime import datetime

In [ ]:
# model_type = YOLO("yolov10m.pt")  # COCO pretrained

In [ ]:
# VEHICLE_CLASSES = {
#     2: "car",
#     3: "motorcycle",
#     5: "bus",
#     7: "truck"
# }



# def detect_vehicles(frame, conf_threshold=0.4):
#     results = model_type(frame, conf=conf_threshold, verbose=False)
    
#     vehicles = []
    
#     for r in results:
#         for box in r.boxes:
#             cls_id = int(box.cls[0])
#             conf = float(box.conf[0])
            
#             if cls_id in VEHICLE_CLASSES:
#                 x1, y1, x2, y2 = map(int, box.xyxy[0])
                
#                 vehicle_data = {
#                     "bbox": [x1, y1, x2, y2],
#                     "vehicle_type": VEHICLE_CLASSES[cls_id],
#                     "confidence": round(conf, 3)
#                 }
                
#                 vehicles.append(vehicle_data)
    
#     return vehicles

In [ ]:
# # Replace with your video path
# video_path = r"C:\Users\100ra\Downloads\Traffic_sound_Indian_Traffic_Sounds_Traffic_Noise_Vehicle_Noise_Road_traffic_Shorts_Viral_cars_360P.mp4"

# cap = cv2.VideoCapture(video_path)

# camera_id = "CAM_01"

# while cap.isOpened():
#     ret, frame = cap.read()
#     if not ret:
#         break

#     vehicles = detect_vehicles(frame)

#     # Draw detections
#     for v in vehicles:
#         x1, y1, x2, y2 = v["bbox"]
#         label = f"{v['vehicle_type']} ({v['confidence']})"

#         cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
#         cv2.putText(frame, label, (x1, y1 - 10),
#                     cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

#         # Example structured log (ready for fusion later)
#         structured_output = {
#             "camera_id": camera_id,
#             "vehicle_type": v["vehicle_type"],
#             "confidence": v["confidence"],
#             "timestamp": datetime.now().isoformat()
#         }

#         print(structured_output)

#     cv2.imshow("Vehicle Type Detection - Video", frame)

#     if cv2.waitKey(1) & 0xFF == ord("q"):
#         break

# cap.release()
# cv2.destroyAllWindows()

## Vehicle color (CNN)

In [ ]:
# import torch
# from torchvision import transforms, models
# import cv2
# from PIL import Image
# import torch.nn as nn

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # Load model
# model_color = models.efficientnet_b0()
# model_color.classifier[1] = nn.Linear(model_color.classifier[1].in_features, 15)

# model_color.load_state_dict(torch.load("vehicle_color_best.pth", map_location=device))
# model_color.to(device)
# model_color.eval()

# color_transform = transforms.Compose([
#     transforms.Resize((224, 224)),
#     transforms.ToTensor(),
# ])

# def predict_vehicle_color(crop):
#     image = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
#     image = Image.fromarray(image)
#     image = color_transform(image).unsqueeze(0).to(device)

#     with torch.no_grad():
#         outputs = model_color(image)
#         _, pred = torch.max(outputs, 1)

#     return class_names[pred.item()]

## Paddle OCR

In [ ]:
# import cv2
# import numpy as np
# import re
# from collections import deque
# from ultralytics import YOLO
# from paddleocr import PaddleOCR


In [ ]:
# model_paddle = YOLO(
#     r"D:\Automatic ANPR\Self_Code\runs\detect\lp_yolov8s_stage2_final3\weights\best.pt"
# )


In [ ]:
# from paddleocr import PaddleOCR

# ocr_engine = PaddleOCR(
#     use_angle_cls=True,
#     lang='en',
#     det=True,
#     rec=True,
#     use_gpu=False  # change if GPU
# )

# def preprocess_plate_for_paddle(plate):
#     # add margin safety
#     h, w = plate.shape[:2]
    
#     # upscale small plates
#     scale = 2.0
#     plate = cv2.resize(plate, (int(w * scale), int(h * scale)))
    
#     # convert to gray
#     gray = cv2.cvtColor(plate, cv2.COLOR_BGR2GRAY)
    
#     # mild CLAHE
#     clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8,8))
#     gray = clahe.apply(gray)
    
#     return gray


# def ocr_plate_paddle(plate_img):
#     result = ocr_engine.ocr(plate_img, cls=True)
    
#     if not result or not result[0]:
#         return ""
    
#     texts = []
    
#     for line in result[0]:
#         text = line[1][0]
#         confidence = line[1][1]
        
#         if confidence > 0.6:
#             texts.append(text)
    
#     if not texts:
#         return ""
    
#     combined = "".join(texts)
#     combined = combined.upper()
#     combined = re.sub(r'[^A-Z0-9]', '', combined)
    
#     return combined

# def normalize_plate_text(text):
#     text = text.upper()
#     text = re.sub(r'[^A-Z0-9]', '', text)
    
#     # common confusion corrections
#     text = text.replace("O", "0")
#     text = text.replace("I", "1")
#     text = text.replace("Z", "2")
#     text = text.replace("S", "5")
    
#     return text


# def validate_indian_plate(text):
#     pattern1 = r'^[A-Z]{2}[0-9]{2}[A-Z]{1}[0-9]{4}$'
#     pattern2 = r'^[A-Z]{2}[0-9]{2}[A-Z]{2}[0-9]{4}$'
    
#     return bool(re.match(pattern1, text) or re.match(pattern2, text))


In [ ]:
# video_path = r"C:\Users\100ra\Downloads\How_High_Security_Number_Plate_challan_system_identifies_car_for_making_challan_digitalautomobile_720P.mp4"
# cap = cv2.VideoCapture(video_path)

# plate_buffer = deque(maxlen=7)
# final_plate = None

# while cap.isOpened():
#     ret, frame = cap.read()
#     if not ret:
#         break
    
#     results = model_paddle(frame, conf=0.4)
    
#     for r in results:
#         for box in r.boxes:
#             x1, y1, x2, y2 = map(int, box.xyxy[0])
            
#             pad = 5
#             x1 = max(0, x1 - pad)
#             y1 = max(0, y1 - pad)
#             x2 = min(frame.shape[1], x2 + pad)
#             y2 = min(frame.shape[0], y2 + pad)
            
#             plate_crop = frame[y1:y2, x1:x2]
            
#             proc = preprocess_plate_for_paddle(plate_crop)
#             raw_text = ocr_plate_paddle(proc)
#             norm_text = normalize_plate_text(raw_text)
            
#             if validate_indian_plate(norm_text):
#                 plate_buffer.append(norm_text)
            
#             if len(plate_buffer) >= 5:
#                 final_plate = max(set(plate_buffer), key=plate_buffer.count)
            
#             cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
            
#             if final_plate:
#                 cv2.putText(frame, final_plate,
#                             (x1, y1-10),
#                             cv2.FONT_HERSHEY_SIMPLEX,
#                             0.8,
#                             (0,255,0),
#                             2)
    
#     cv2.imshow("PaddleOCR ANPR", frame)
    
#     if cv2.waitKey(1) & 0xFF == ord('q'):
#         break

# cap.release()
# cv2.destroyAllWindows()


# **Combining all (Type+color+OCR)**

Load All Models

In [1]:
from ultralytics import YOLO
import cv2
import numpy as np
from collections import defaultdict, deque
from datetime import datetime
import torch
from torchvision import transforms, models
import torch.nn as nn
from PIL import Image
from paddleocr import PaddleOCR
import re

# ---------------------------------------------------
# CONFIG
# ---------------------------------------------------

VEHICLE_CLASSES = {
    2: "car",
    3: "motorcycle",
    5: "bus",
    7: "truck"
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------------------------------
# LOAD MODELS
# ---------------------------------------------------

# Vehicle detection + tracking
model_type = YOLO("yolov10m.pt")

# Plate detector (your trained model)
model_plate = YOLO(
    r"D:\Automatic ANPR\Self_Code\runs\detect\lp_yolov8s_stage2_final3\weights\best.pt"
)

# Color model (EfficientNet trained by you)
model_color = models.efficientnet_b0()
model_color.classifier[1] = nn.Linear(model_color.classifier[1].in_features, 15)
model_color.load_state_dict(torch.load("vehicle_color_best.pth", map_location=device))
model_color.to(device)
model_color.eval()

class_names = ['beige','black','blue','brown','gold','green','grey','orange',
               'pink','purple','red','silver','tan','white','yellow']

color_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Paddle OCR
ocr_engine = PaddleOCR(
    use_angle_cls=True,
    lang='en',
    det=True,
    rec=True,
    use_gpu=False  # change if GPU
)

# ---------------------------------------------------
# HELPER FUNCTIONS
# ---------------------------------------------------

def predict_vehicle_color(crop):
    image = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    image = Image.fromarray(image)
    image = color_transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model_color(image)
        _, pred = torch.max(outputs, 1)

    return class_names[pred.item()]


def preprocess_plate_for_paddle(plate):
    h, w = plate.shape[:2]
    plate = cv2.resize(plate, (int(w * 2), int(h * 2)))
    gray = cv2.cvtColor(plate, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8,8))
    return clahe.apply(gray)


def ocr_plate_paddle(plate_img):
    result = ocr_engine.ocr(plate_img, cls=True)

    if not result or not result[0]:
        return ""

    texts = []
    for line in result[0]:
        text = line[1][0]
        conf = line[1][1]
        if conf > 0.6:
            texts.append(text)

    combined = "".join(texts).upper()
    combined = re.sub(r'[^A-Z0-9]', '', combined)
    return combined


def validate_indian_plate(text):
    pattern1 = r'^[A-Z]{2}[0-9]{2}[A-Z]{1}[0-9]{4}$'
    pattern2 = r'^[A-Z]{2}[0-9]{2}[A-Z]{2}[0-9]{4}$'
    return bool(re.match(pattern1, text) or re.match(pattern2, text))


# ---------------------------------------------------
# VIDEO PIPELINE
# ---------------------------------------------------

video_path = r"C:\Users\100ra\Downloads\Toll plaza automatic vehicle recognition.mp4"
cap = cv2.VideoCapture(0)

track_memory = defaultdict(lambda: {
    "plate_buffer": deque(maxlen=7),
    "final_plate": "",
    "color": "",
    "type": ""
})

while cap.isOpened():

    ret, frame = cap.read()
    if not ret:
        break

    results = model_type.track(frame, persist=True, verbose=False)

    if results[0].boxes.id is None:
        cv2.imshow("ANPR System", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        continue

    boxes = results[0].boxes.xyxy.cpu().numpy()
    classes = results[0].boxes.cls.cpu().numpy()
    ids = results[0].boxes.id.cpu().numpy()

    for box, cls, track_id in zip(boxes, classes, ids):

        cls = int(cls)
        track_id = int(track_id)

        if cls not in VEHICLE_CLASSES:
            continue

        x1, y1, x2, y2 = map(int, box)
        vehicle_crop = frame[y1:y2, x1:x2]

        if vehicle_crop.size == 0:
            continue

        # ---------------- VEHICLE TYPE ----------------
        vehicle_type = VEHICLE_CLASSES[cls]
        track_memory[track_id]["type"] = vehicle_type

        # ---------------- COLOR ----------------
        vehicle_color = predict_vehicle_color(vehicle_crop)
        track_memory[track_id]["color"] = vehicle_color

        # ---------------- PLATE DETECTION ----------------
        plate_results = model_plate(vehicle_crop, conf=0.4, verbose=False)

        for pr in plate_results:
            for pbox in pr.boxes:

                px1, py1, px2, py2 = map(int, pbox.xyxy[0])
                plate_crop = vehicle_crop[py1:py2, px1:px2]

                if plate_crop.size == 0:
                    continue

                proc = preprocess_plate_for_paddle(plate_crop)
                raw_text = ocr_plate_paddle(proc)

                if validate_indian_plate(raw_text):
                    track_memory[track_id]["plate_buffer"].append(raw_text)

        # ---------------- TEMPORAL VOTING ----------------
        buffer = track_memory[track_id]["plate_buffer"]
        if len(buffer) >= 5:
            final_plate = max(set(buffer), key=buffer.count)
            track_memory[track_id]["final_plate"] = final_plate

        # ---------------- DRAW ----------------
        label = f"ID:{track_id} {vehicle_type} {vehicle_color}"

        if track_memory[track_id]["final_plate"]:
            label += f" {track_memory[track_id]['final_plate']}"

        cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
        cv2.putText(frame, label,
                    (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0,255,0), 2)

    cv2.imshow("ANPR System", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

[2026/02/16 10:19:25] ppocr DEBUG: Namespace(help='==SUPPRESS==', use_gpu=False, use_xpu=False, use_npu=False, ir_optim=True, use_tensorrt=False, min_subgraph_size=15, precision='fp32', gpu_mem=500, image_dir=None, page_num=0, det_algorithm='DB', det_model_dir='C:\\Users\\100ra/.paddleocr/whl\\det\\en\\en_PP-OCRv3_det_infer', det_limit_side_len=960, det_limit_type='max', det_box_type='quad', det_db_thresh=0.3, det_db_box_thresh=0.6, det_db_unclip_ratio=1.5, max_batch_size=10, use_dilation=False, det_db_score_mode='fast', det_east_score_thresh=0.8, det_east_cover_thresh=0.1, det_east_nms_thresh=0.2, det_sast_score_thresh=0.5, det_sast_nms_thresh=0.2, det_pse_thresh=0, det_pse_box_thresh=0.85, det_pse_min_area=16, det_pse_scale=1, scales=[8, 16, 32], alpha=1.0, beta=1.0, fourier_degree=5, rec_algorithm='SVTR_LCNet', rec_model_dir='C:\\Users\\100ra/.paddleocr/whl\\rec\\en\\en_PP-OCRv3_rec_infer', rec_image_inverse=True, rec_image_shape='3, 48, 320', rec_batch_num=6, max_text_length=25, re

## FINAL OPTIMIZED CODE (YOUR PIPELINE + ROI + FREEZE)


In [2]:
from ultralytics import YOLO
import cv2
import numpy as np
from collections import defaultdict, deque
import torch
from torchvision import transforms, models
import torch.nn as nn
from PIL import Image
from paddleocr import PaddleOCR
import re

# ---------------------------------------------------
# CONFIG
# ---------------------------------------------------

VEHICLE_CLASSES = {
    2: "car",
    3: "motorcycle",
    5: "bus",
    7: "truck"
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------------------------------
# LOAD MODELS
# ---------------------------------------------------

model_type = YOLO("yolov10m.pt")

model_plate = YOLO(
    r"D:\Automatic ANPR\Self_Code\runs\detect\lp_yolov8s_stage2_final3\weights\best.pt"
)

model_color = models.efficientnet_b0()
model_color.classifier[1] = nn.Linear(model_color.classifier[1].in_features, 15)
model_color.load_state_dict(torch.load("vehicle_color_best.pth", map_location=device))
model_color.to(device)
model_color.eval()

class_names = ['beige','black','blue','brown','gold','green','grey','orange',
               'pink','purple','red','silver','tan','white','yellow']

color_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

ocr_engine = PaddleOCR(use_angle_cls=True, lang='en')

# ---------------------------------------------------
# ROI FUNCTION
# ---------------------------------------------------

def inside_roi(bbox, frame_h):
    x1, y1, x2, y2 = bbox
    cy = (y1 + y2) // 2
    
    ROI_Y_MIN = int(frame_h * 0.35)
    ROI_Y_MAX = int(frame_h * 0.85)
    
    return ROI_Y_MIN <= cy <= ROI_Y_MAX

# ---------------------------------------------------
# HELPERS
# ---------------------------------------------------


def predict_vehicle_color(crop):
    image = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    image = Image.fromarray(image)
    image = color_transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        outputs = model_color(image)
        _, pred = torch.max(outputs, 1)
    return class_names[pred.item()]

def preprocess_plate_for_paddle(plate):
    h, w = plate.shape[:2]
    plate = cv2.resize(plate, (int(w * 2), int(h * 2)))
    gray = cv2.cvtColor(plate, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8,8))
    return clahe.apply(gray)

def ocr_plate_paddle(img):
    result = ocr_engine.ocr(img, cls=True)
    if not result or not result[0]:
        return ""
    texts = []
    for line in result[0]:
        t, conf = line[1]
        if conf > 0.6:
            texts.append(t)
    text = "".join(texts).upper()
    return re.sub(r'[^A-Z0-9]', '', text)

def validate_indian_plate(text):
    p1 = r'^[A-Z]{2}[0-9]{2}[A-Z]{1}[0-9]{4}$'
    p2 = r'^[A-Z]{2}[0-9]{2}[A-Z]{2}[0-9]{4}$'
    return bool(re.match(p1, text) or re.match(p2, text))

# ---------------------------------------------------
# VIDEO
# ---------------------------------------------------

video_path = r"C:\Users\100ra\Downloads\ANPR India Detection Demo - SmartCow.mp4"
cap = cv2.VideoCapture(video_path)

track_memory = defaultdict(lambda: {
    "state": "idle",
    "plate_buffer": deque(maxlen=7),
    "final_plate": "",
    "color": "",
    "type": ""
})

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    H = frame.shape[0]

    results = model_type.track(frame, persist=True, verbose=False)

    if results[0].boxes.id is None:
        cv2.imshow("ANPR System", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        continue

    boxes = results[0].boxes.xyxy.cpu().numpy()
    classes = results[0].boxes.cls.cpu().numpy()
    ids = results[0].boxes.id.cpu().numpy()

    for box, cls, track_id in zip(boxes, classes, ids):

        cls = int(cls)
        track_id = int(track_id)

        if cls not in VEHICLE_CLASSES:
            continue

        x1, y1, x2, y2 = map(int, box)
        bbox = [x1, y1, x2, y2]

        mem = track_memory[track_id]

        # ---------------- ROI CHECK ----------------
        if not inside_roi(bbox, H):
            continue

        # If already done → skip heavy work
        if mem["state"] == "done":
            label = f"ID:{track_id} {mem['type']} {mem['color']} {mem['final_plate']}"
            cv2.rectangle(frame,(x1,y1),(x2,y2),(0,255,0),2)
            cv2.putText(frame,label,(x1,y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,255,0),2)
            continue

        # Activate track
        mem["state"] = "active"

        vehicle_crop = frame[y1:y2, x1:x2]
        if vehicle_crop.size == 0:
            continue

        # TYPE
        mem["type"] = VEHICLE_CLASSES[cls]

        # COLOR
        if mem["color"] == "":
            mem["color"] = predict_vehicle_color(vehicle_crop)

        # PLATE DETECT
        plate_results = model_plate(vehicle_crop, conf=0.4, verbose=False)

        for pr in plate_results:
            for pbox in pr.boxes:
                px1, py1, px2, py2 = map(int, pbox.xyxy[0])
                plate_crop = vehicle_crop[py1:py2, px1:px2]
                if plate_crop.size == 0:
                    continue

                proc = preprocess_plate_for_paddle(plate_crop)
                raw = ocr_plate_paddle(proc)

                if validate_indian_plate(raw):
                    mem["plate_buffer"].append(raw)

        # TEMPORAL VOTING
        buf = mem["plate_buffer"]
        if len(buf) >= 5:
            mem["final_plate"] = max(set(buf), key=buf.count)
            mem["state"] = "done"   # FREEZE

        # DRAW
        label = f"ID:{track_id} {mem['type']} {mem['color']}"
        if mem["final_plate"]:
            label += f" {mem['final_plate']}"

        cv2.rectangle(frame,(x1,y1),(x2,y2),(0,255,0),2)
        cv2.putText(frame,label,(x1,y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,255,0),2)

    cv2.imshow("ANPR System", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

[2026/02/17 16:00:25] ppocr DEBUG: Namespace(help='==SUPPRESS==', use_gpu=False, use_xpu=False, use_npu=False, ir_optim=True, use_tensorrt=False, min_subgraph_size=15, precision='fp32', gpu_mem=500, image_dir=None, page_num=0, det_algorithm='DB', det_model_dir='C:\\Users\\100ra/.paddleocr/whl\\det\\en\\en_PP-OCRv3_det_infer', det_limit_side_len=960, det_limit_type='max', det_box_type='quad', det_db_thresh=0.3, det_db_box_thresh=0.6, det_db_unclip_ratio=1.5, max_batch_size=10, use_dilation=False, det_db_score_mode='fast', det_east_score_thresh=0.8, det_east_cover_thresh=0.1, det_east_nms_thresh=0.2, det_sast_score_thresh=0.5, det_sast_nms_thresh=0.2, det_pse_thresh=0, det_pse_box_thresh=0.85, det_pse_min_area=16, det_pse_scale=1, scales=[8, 16, 32], alpha=1.0, beta=1.0, fourier_degree=5, rec_algorithm='SVTR_LCNet', rec_model_dir='C:\\Users\\100ra/.paddleocr/whl\\rec\\en\\en_PP-OCRv3_rec_infer', rec_image_inverse=True, rec_image_shape='3, 48, 320', rec_batch_num=6, max_text_length=25, re

## Color

In [7]:
import torch
from torchvision import transforms, models
import cv2
from PIL import Image
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load model
model_color = models.efficientnet_b0()
model_color.classifier[1] = nn.Linear(model_color.classifier[1].in_features, 15)

model_color.load_state_dict(torch.load("vehicle_color_best.pth", map_location=device))
model_color.to(device)
model_color.eval()

color_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

def predict_vehicle_color(crop):
    image = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    image = Image.fromarray(image)
    image = color_transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model_color(image)
        _, pred = torch.max(outputs, 1)

    return class_names[pred.item()]


img = r""
crop = cv2.imread(img)
color = predict_vehicle_color(crop) 
print("Predicted Color:", color)

FileNotFoundError: [Errno 2] No such file or directory: ''